# 15 - Gold: Dashboard Panel Marts

This notebook creates panel-specific gold tables for the first Streamlit dashboard.

Outputs:

- `gold.dashboard_country_timeseries`
- `gold.dashboard_top_trade_partners`
- `gold.dashboard_conflict_hotspots`
- `gold.dashboard_fragility_components`
- `gold.dashboard_bloc_comparison`

The previous gold notebook created broad, reusable observatory marts. This notebook narrows those data into shapes that map directly to dashboard components.

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

TOP_N_PARTNERS = 15
HOTSPOT_LOOKBACK_YEARS = 3
loaded_at = datetime.now(timezone.utc)

print("Target panel marts:")
print("  gold.dashboard_country_timeseries")
print("  gold.dashboard_top_trade_partners")
print("  gold.dashboard_conflict_hotspots")
print("  gold.dashboard_fragility_components")
print("  gold.dashboard_bloc_comparison")

In [ ]:
# Cell 2 - Input table checks
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0


required_tables = [
    ("gold", "country_year_observatory"),
    ("gold", "country_latest_snapshot"),
    ("gold", "bloc_year_observatory"),
    ("gold", "bloc_latest_snapshot"),
    ("silver", "fact_trade_partner_annual"),
    ("silver", "fact_acled_weekly"),
    ("silver", "fact_fsi_annual"),
]

missing_tables = [f"{schema}.{table}" for schema, table in required_tables if not table_exists(schema, table)]
if missing_tables:
    raise ValueError(f"Missing required input table(s): {missing_tables}")

print("All required tables found.")
for schema_name, table_name in required_tables:
    print(f"  {schema_name}.{table_name}: {spark.table(f'{schema_name}.{table_name}').count():,} rows")

In [ ]:
# Cell 3 - Country time-series panel mart
country_timeseries_df = (
    spark.table("gold.country_year_observatory")
    .select(
        "country_iso3",
        "country_name",
        "project_region",
        "analytical_bloc_code",
        "analytical_bloc_name",
        "year",
        "coverage_level",
        "dashboard_core_ready",
        "full_fragility_context_ready",
        "gdp_current_usd_billions",
        "gdp_per_capita_current_usd",
        "population_millions",
        "real_gdp_growth_pct_imf",
        "inflation_cpi_pct",
        "gross_debt_pct_gdp_imf",
        "net_lending_borrowing_pct_gdp_imf",
        "current_account_balance_pct_gdp_imf",
        "exports_billions_usd",
        "imports_billions_usd",
        "total_trade_billions_usd",
        "trade_balance_billions_usd",
        "trade_openness_pct_gdp",
        "total_trade_partner_hhi",
        "top_partner_iso3",
        "top_partner_name",
        "top_partner_share_pct",
        "total_events",
        "violent_events",
        "fatalities",
        "events_per_million",
        "violent_events_per_million",
        "fatalities_per_million",
        "fsi_total_score",
        "fragility_band",
        "cohesion_score",
        "economic_score",
        "political_score",
        "social_cross_cutting_score",
    )
    .withColumn("source_refresh_at", F.lit(loaded_at))
)

spark.sql("DROP TABLE IF EXISTS gold.dashboard_country_timeseries")
(country_timeseries_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dashboard_country_timeseries"))

print("Write complete: gold.dashboard_country_timeseries")
country_timeseries_df.orderBy("country_iso3", "year").show(10, truncate=False)

In [ ]:
# Cell 4 - Top trade partners panel mart
partner_rank_window = Window.partitionBy("reporter_iso3", "year").orderBy(F.desc("total_trade_usd"), F.asc("counterpart_iso3"))

top_trade_partners_df = (
    spark.table("silver.fact_trade_partner_annual")
    .where(~F.col("is_world_aggregate"))
    .withColumn("partner_rank", F.row_number().over(partner_rank_window))
    .where(F.col("partner_rank") <= TOP_N_PARTNERS)
    .select(
        F.col("reporter_iso3").alias("country_iso3"),
        F.col("reporter_name").alias("country_name"),
        F.col("reporter_bloc_code").alias("analytical_bloc_code"),
        F.col("reporter_bloc_name").alias("analytical_bloc_name"),
        "year",
        "partner_rank",
        "counterpart_iso3",
        "counterpart_name",
        "counterpart_is_project_country",
        "counterpart_group_code",
        "counterpart_group_name",
        "exports_billions_usd",
        "imports_billions_usd",
        "total_trade_billions_usd",
        "trade_balance_billions_usd",
        "export_partner_share_pct",
        "import_partner_share_pct",
        "total_trade_partner_share_pct",
    )
    .withColumn("source_refresh_at", F.lit(loaded_at))
)

spark.sql("DROP TABLE IF EXISTS gold.dashboard_top_trade_partners")
(top_trade_partners_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dashboard_top_trade_partners"))

print("Write complete: gold.dashboard_top_trade_partners")
top_trade_partners_df.where(F.col("year") == F.col("year")).orderBy("country_iso3", "year", "partner_rank").show(20, truncate=False)

In [ ]:
# Cell 5 - Conflict hotspot panel mart
max_acled_year = spark.table("silver.fact_acled_weekly").agg(F.max("year")).first()[0]
hotspot_start_year = max_acled_year - HOTSPOT_LOOKBACK_YEARS + 1
print(f"Conflict hotspot window: {hotspot_start_year}-{max_acled_year}")

hotspot_rank_window = Window.partitionBy("country_iso3").orderBy(F.desc("violent_events"), F.desc("fatalities"), F.asc("admin1"))

conflict_hotspots_df = (
    spark.table("silver.fact_acled_weekly")
    .where(F.col("year") >= hotspot_start_year)
    .groupBy("country_iso3", "country_name", "analytical_bloc_code", "analytical_bloc_name", "admin1")
    .agg(
        F.min("year").alias("window_start_year"),
        F.max("year").alias("window_end_year"),
        F.sum("event_count").alias("total_events"),
        F.sum(F.when(F.col("is_violent_event"), F.col("event_count")).otherwise(F.lit(0))).alias("violent_events"),
        F.sum("fatalities").alias("fatalities"),
        F.sum(F.when(F.col("event_type") == "Battles", F.col("event_count")).otherwise(F.lit(0))).alias("battle_events"),
        F.sum(F.when(F.col("event_type") == "Violence against civilians", F.col("event_count")).otherwise(F.lit(0))).alias("violence_against_civilians_events"),
        F.sum(F.when(F.col("event_type") == "Explosions/Remote violence", F.col("event_count")).otherwise(F.lit(0))).alias("explosions_remote_violence_events"),
        F.sum(F.when(F.col("event_type") == "Protests", F.col("event_count")).otherwise(F.lit(0))).alias("protest_events"),
        F.sum(F.when(F.col("event_type") == "Riots", F.col("event_count")).otherwise(F.lit(0))).alias("riot_events"),
        F.max("population").alias("latest_population"),
    )
    .withColumn("events_per_million", F.when(F.col("latest_population") > 0, F.col("total_events") / F.col("latest_population") * 1_000_000.0))
    .withColumn("fatalities_per_million", F.when(F.col("latest_population") > 0, F.col("fatalities") / F.col("latest_population") * 1_000_000.0))
    .withColumn("hotspot_rank", F.row_number().over(hotspot_rank_window))
    .withColumn("source_refresh_at", F.lit(loaded_at))
)

spark.sql("DROP TABLE IF EXISTS gold.dashboard_conflict_hotspots")
(conflict_hotspots_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dashboard_conflict_hotspots"))

print("Write complete: gold.dashboard_conflict_hotspots")
conflict_hotspots_df.where(F.col("hotspot_rank") <= 5).orderBy("country_iso3", "hotspot_rank").show(80, truncate=False)

In [ ]:
# Cell 6 - Fragility components panel mart
latest_fsi_year_by_country = (
    spark.table("silver.fact_fsi_annual")
    .groupBy("country_iso3")
    .agg(F.max("year").alias("latest_fsi_year"))
)

fragility_components_df = (
    spark.table("silver.fact_fsi_annual").alias("f")
    .join(latest_fsi_year_by_country.alias("y"), (F.col("f.country_iso3") == F.col("y.country_iso3")) & (F.col("f.year") == F.col("y.latest_fsi_year")), "inner")
    .select(
        F.col("f.country_iso3"),
        F.col("f.country_name"),
        F.col("f.analytical_bloc_code"),
        F.col("f.analytical_bloc_name"),
        F.col("f.year").alias("fsi_year"),
        F.col("f.fsi_rank"),
        F.col("f.fsi_total_score"),
        F.col("f.fragility_band"),
        F.col("f.cohesion_score"),
        F.col("f.economic_score"),
        F.col("f.political_score"),
        F.col("f.social_cross_cutting_score"),
        F.col("f.security_apparatus_score"),
        F.col("f.factionalized_elites_score"),
        F.col("f.group_grievance_score"),
        F.col("f.economic_decline_score"),
        F.col("f.uneven_economic_development_score"),
        F.col("f.human_flight_brain_drain_score"),
        F.col("f.state_legitimacy_score"),
        F.col("f.public_services_score"),
        F.col("f.human_rights_rule_of_law_score"),
        F.col("f.demographic_pressures_score"),
        F.col("f.refugees_idps_score"),
        F.col("f.external_intervention_score"),
    )
    .withColumn("source_refresh_at", F.lit(loaded_at))
)

spark.sql("DROP TABLE IF EXISTS gold.dashboard_fragility_components")
(fragility_components_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dashboard_fragility_components"))

print("Write complete: gold.dashboard_fragility_components")
fragility_components_df.orderBy(F.desc("fsi_total_score")).show(25, truncate=False)

In [ ]:
# Cell 7 - Bloc comparison panel mart
bloc_comparison_df = (
    spark.table("gold.bloc_year_observatory")
    .select(
        "analytical_bloc_code",
        "analytical_bloc_name",
        "year",
        "country_count",
        "gdp_current_usd_billions",
        "population_millions",
        "bloc_gdp_per_capita_current_usd",
        "exports_billions_usd",
        "imports_billions_usd",
        "total_trade_billions_usd",
        "trade_balance_billions_usd",
        "trade_openness_pct_gdp",
        "gross_debt_pct_gdp",
        "violent_events",
        "fatalities",
        "violent_events_per_million",
        "fatalities_per_million",
        "avg_fsi_total_score",
        "avg_total_trade_partner_hhi",
        "dashboard_core_ready_countries",
        "full_context_ready_countries",
    )
    .withColumn("source_refresh_at", F.lit(loaded_at))
)

spark.sql("DROP TABLE IF EXISTS gold.dashboard_bloc_comparison")
(bloc_comparison_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dashboard_bloc_comparison"))

print("Write complete: gold.dashboard_bloc_comparison")
bloc_comparison_df.orderBy("year", "analytical_bloc_code").show(30, truncate=False)

In [ ]:
# Cell 8 - Verification and dashboard sanity checks
panel_tables = [
    "dashboard_country_timeseries",
    "dashboard_top_trade_partners",
    "dashboard_conflict_hotspots",
    "dashboard_fragility_components",
    "dashboard_bloc_comparison",
]

print("Panel mart row counts:")
for table_name in panel_tables:
    row_count = spark.table(f"gold.{table_name}").count()
    print(f"  gold.{table_name}: {row_count:,}")

print("Latest country time-series coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS latest_country_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        SUM(CASE WHEN dashboard_core_ready THEN 1 ELSE 0 END) AS dashboard_core_ready_countries,
        SUM(CASE WHEN fsi_total_score IS NOT NULL THEN 1 ELSE 0 END) AS current_year_fsi_countries
    FROM gold.dashboard_country_timeseries
    WHERE year = (SELECT MAX(year) FROM gold.dashboard_country_timeseries)
""").show()

print("Latest top partners for Cameroon:")
spark.sql("""
    SELECT
        year,
        partner_rank,
        counterpart_iso3,
        counterpart_name,
        ROUND(total_trade_billions_usd, 2) AS total_trade_billions_usd,
        ROUND(total_trade_partner_share_pct, 1) AS total_trade_share_pct
    FROM gold.dashboard_top_trade_partners
    WHERE country_iso3 = 'CMR'
      AND year = (SELECT MAX(year) FROM gold.dashboard_top_trade_partners)
    ORDER BY partner_rank
""").show(20, truncate=False)

print("Top conflict hotspots, latest window:")
spark.sql("""
    SELECT
        country_iso3,
        country_name,
        admin1,
        hotspot_rank,
        total_events,
        violent_events,
        fatalities
    FROM gold.dashboard_conflict_hotspots
    WHERE hotspot_rank <= 3
    ORDER BY country_iso3, hotspot_rank
""").show(80, truncate=False)

print("Latest bloc comparison:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        year,
        country_count,
        ROUND(gdp_current_usd_billions, 2) AS gdp_billions_usd,
        ROUND(total_trade_billions_usd, 2) AS trade_billions_usd,
        ROUND(trade_openness_pct_gdp, 1) AS trade_openness_pct_gdp,
        violent_events,
        fatalities,
        ROUND(avg_fsi_total_score, 1) AS avg_fsi_total_score
    FROM gold.dashboard_bloc_comparison
    WHERE year = (SELECT MAX(year) FROM gold.dashboard_bloc_comparison)
    ORDER BY analytical_bloc_code
""").show(truncate=False)